# Public historical notebook
Outputs, authentication metadata and personal artifact links have been removed for publication.
This is a historical code record, NOT a complete retraining kit. Do not Run All.
See the stage README and aggregate training history. Private datasets and model archives are not included.


# Colab arxiv nusxasi

Bu VS Code’da ko‘rsatish uchun saqlangan tarixiy notebook. Bu ishga tushirish shabloni; bajarilgan tarix deb ko‘rsatilmaydi. Maxfiy tokenlar yashiriladi, HTML/rasm chiqishlari olinmaydi. Treningni laptopda Run All qilmang. Haqiqiy yangi trening uchun Colab va alohida run kerak.


# UzNorm v2 — ByT5-Base — Colab full fine-tuning

206 016 juftlik: 171 845 train / 16 708 validation / 17 463 test.
Bu targetlar inson tasdiqlagan Gold emas. Trening yangi `google/byt5-base` checkpointidan boshlanadi;
eski v1 runiga yangi data bilan resume qilinmaydi. Bu notebook training kodini patch qilmaydi.

Avval Drive/MyDrive/uznorm/ ichiga **uznorm-byt5-base-safe-v2.zip** va
**uznorm-byt5-base-safe-v2.delivery.json** ni yuklang. Runtime → Change runtime type → GPU.
Colab Secrets ichida `WANDB_API_KEY` yarating va Notebook access’ni yoqing. Kalitni kodga yozmang.
Uyali oynalarni ketma-ket bajaring; full trening va test alohida ruxsat bayroqlari bilan yopilgan.

Base: LR=5e-5, micro-batch=1, effective batch=32, eval batch=1, 3 epochgacha. Bu ehtiyotkor boshlang‘ich sozlama, optimal yoki 100 unitga sig‘ishi kafolatlanmagan. Native BF16 bo‘lsa ishlatiladi, aks holda FP32; TF32/FP16 o‘chiq. Small smoke/checkpoint bu run uchun ishlatilmaydi. Eski runni parallel boshlamang. Avvalgi noto‘liq qat’iy model almashtirish patchlari kerak emas: yangi notebookni oching.


In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
if not os.path.ismount('/content/drive'):
    raise RuntimeError('Drive haqiqiy mount emas. Treningni boshlamang.')

## 1. Paket yaxlitligini tekshirish va alohida lokal papkaga ochish

ZIP va delivery fayli bir release’dan bo‘lishi shart. Bu qadam eski Drive dataset/runlarini o‘zgartirmaydi.
Paket Drive/uznorm yoki /content papkasidan avtomatik topiladi.
Boshqa joyga yuklagan bo‘lsangiz faqat `PACKAGE_DIR`ni moslang; fayl nomlarini o‘zgartirmang.

In [ ]:
from pathlib import Path
import hashlib, json, os, stat, subprocess, sys, tempfile, uuid, zipfile

PACKAGE_DIR = None  # Masalan: Path('/content'); None bo‘lsa ikki odatiy papkadan qidiradi.
def locate_package(folders, archive_name):
    for folder in folders:
        candidate = Path(folder) / archive_name
        delivery = candidate.with_suffix('.delivery.json')
        if candidate.is_file() and delivery.is_file():
            return candidate, delivery
    raise FileNotFoundError(
        f'{archive_name} VA unga mos .delivery.json bir papkada topilmadi. '
        'Ikkalasini MyDrive/uznorm/ yoki Colab Files orqali /content/ ichiga yuklang.')
folders = [PACKAGE_DIR] if PACKAGE_DIR is not None else [Path('/content/drive/MyDrive/uznorm'), Path('/content')]
ZIP_PATH, DELIVERY_PATH = locate_package(folders, 'uznorm-byt5-base-safe-v2.zip')
print('Paket:', ZIP_PATH)
def file_sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()
delivery = json.loads(DELIVERY_PATH.read_text(encoding='utf-8'))
if file_sha(ZIP_PATH) != delivery['sha256']:
    raise ValueError('ZIP hash mos emas. ZIP va delivery faylini bir xil release’dan qayta yuklang.')
PROJECT = Path(tempfile.mkdtemp(prefix='uznorm-v2-', dir='/content'))
with zipfile.ZipFile(ZIP_PATH) as archive:
    members = archive.infolist()
    if len({m.filename for m in members}) != len(members):
        raise ValueError('Duplicate ZIP paths')
    if sum(m.file_size for m in members) > 8 * 1024**3:
        raise ValueError('Unexpectedly large archive')
    for member in members:
        relative = Path(member.filename)
        target = (PROJECT / relative).resolve()
        if (relative.is_absolute() or '..' in relative.parts or ':' in member.filename
            or chr(92) in member.filename or not target.is_relative_to(PROJECT)
            or stat.S_ISLNK(member.external_attr >> 16)):
            raise ValueError('Unsafe ZIP path')
    archive.extractall(PROJECT)
manifest = json.loads((PROJECT / 'release-manifest.json').read_text(encoding='utf-8'))
for name, expected in manifest['files'].items():
    path = (PROJECT / name).resolve()
    if not path.is_relative_to(PROJECT) or file_sha(path) != expected:
        raise ValueError('Release file mismatch')
print('Paket tekshirildi:', PROJECT)

## 2. Kutubxonalar va log chiqaruvchi yordamchi

CUDA’li `torch` qayta o‘rnatilmaydi. Yangi runtime’da shu katakni ML importlaridan oldin bajaring.
Kutubxonalar avval import qilingan bo‘lsa install’dan keyin runtime’ni qayta ishga tushiring.

In [ ]:
os.environ.update(USE_TF='0', USE_FLAX='0', WANDB_LOG_MODEL='false', WANDB_WATCH='false',
                  WANDB_DISABLE_CODE='true', WANDB_CONSOLE='off', TOKENIZERS_PARALLELISM='false')
def run_command(arguments, log_path=None):
    secrets = [os.environ.get(key, '') for key in ('WANDB_API_KEY', 'HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN')]
    handle = None
    if log_path:
        from uznorm.storage import guard_output
        guard_output(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        handle = log_path.open('a', encoding='utf-8')
    process = subprocess.Popen(arguments, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding='utf-8', errors='replace', bufsize=1)
    try:
        for line in process.stdout:
            for secret in secrets:
                if secret:
                    line = line.replace(secret, '[REDACTED]')
            print(line, end='', flush=True)
            if handle:
                handle.write(line)
                handle.flush()
        code = process.wait()
        if code:
            raise RuntimeError(f'Jarayon to‘xtadi (exit={code}). Yuqoridagi xatoni ko‘ring; kalitni yubormang.')
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        process.stdout.close()
        if handle:
            handle.close()
run_command([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements-colab.txt')])
run_command([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(PROJECT)])
# Editable .pth fayli yangi subprocess’da o‘qiladi, lekin mavjud kernel’da avtomatik yangilanmaydi.
# Oldingi Small paketidan qolgan importlarni yangi, hash-tekshirilgan paket bilan almashtiramiz.
import importlib
for name in list(sys.modules):
    if name == 'uznorm' or name.startswith('uznorm.'):
        del sys.modules[name]
source_dir = str(PROJECT / 'src')
sys.path.insert(0, source_dir)
os.environ['PYTHONPATH'] = source_dir + (os.pathsep + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')
importlib.invalidate_caches()
from uznorm.io import read_json, write_json
import uznorm
if Path(uznorm.__file__).resolve().parent != (PROJECT / 'src/uznorm').resolve():
    raise RuntimeError('Notebook boshqa uznorm paketini yukladi. Yangi runtime’dan boshlang.')
print('Notebook va CLI paketi tayyor:', uznorm.__version__)

## Drive saqlash himoyasi — bir marta hisob/papkani tasdiqlang

Colab Pro ham uzluksiz sessiyani kafolatlamaydi. Bu kod sessiya uzilishini emas,
vaqtinchalik papkaga jim yozib yuborishni oldini olish va checkpointdan tiklanish uchun.
Birinchi ishga tushirishda `uznorm/.uznorm-storage.json` fayli Drive’da yaratiladi.
**Drive veb-sahifasida aynan trening uchun tanlangan hisobdan shu faylni oching**;
ichidagi `storage_id`ni pastdagi `EXPECTED_STORAGE_ID`ga qo‘ying va shu katakni qayta bajaring.
Notebookni saqlang: keyingi sessiyada shu ID o‘zgarmasin. ID API kaliti emas.
Bu kichik faylning Drive’da ko‘rinishini tasdiqlaydi; keyingi katta checkpointlar sync’iga kafolat emas.

In [ ]:
from uznorm.storage import prepare_storage, guard_output
STORAGE_ROOT = Path('/content/drive/MyDrive/uznorm')
EXPECTED_STORAGE_ID = ''  # Drive veb-sahifasidagi .uznorm-storage.json faylidan kiriting.
os.environ['UZNORM_REQUIRE_DRIVE'] = '1'
storage = prepare_storage(STORAGE_ROOT, EXPECTED_STORAGE_ID)
print('Storage fayli:', storage['marker'])
if not EXPECTED_STORAGE_ID:
    os.environ.pop('UZNORM_STORAGE_ID', None)
    print('PAUZA: Drive veb-sahifasidan storage_id ni yuqoriga kiriting va katakni qayta bajaring.')
else:
    os.environ['UZNORM_STORAGE_ID'] = EXPECTED_STORAGE_ID
    os.environ['UZNORM_STORAGE_ROOT'] = str(STORAGE_ROOT)
    print('Drive mount, storage ID va yozish/o‘qish tekshiruvi o‘tdi.')

## 3. Config va W&B

Yangi tajriba uchun yangi `RUN_NAME`. Model/config/data/batch/precision/runtime o‘zgarsa eski checkpointdan
resume taqiqlanadi. Config o‘zgartirmoqchi bo‘lsangiz nusxani tahrirlang, `RUN_NAME`ni almashtiring.

In [ ]:
from google.colab import userdata
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY').strip()
except Exception:
    raise RuntimeError('Colab Secrets: WANDB_API_KEY qo‘shing va Notebook access’ni yoqing.') from None
if not os.environ['WANDB_API_KEY']:
    raise RuntimeError('WANDB_API_KEY bo‘sh')

RUN_NAME = 'byt5-base-v2-safe1'
RUN_BASE = Path('/content/drive/MyDrive/uznorm/runs-v2')
RUN_DIR = RUN_BASE / RUN_NAME
CONFIG_PATH = PROJECT / 'configs/byt5-base-v2.json'
guard_output(RUN_DIR)
chosen_config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
if chosen_config['model']['name'] != 'google/byt5-base':
    raise RuntimeError('Notebook/model mos emas. Trening boshlanmaydi.')
print('MODEL:', chosen_config['model']['name'])
print('SAQLASH JOYI:', RUN_DIR)
print('STORAGE ID:', EXPECTED_STORAGE_ID)
print('DATASET SHA:', chosen_config['data']['manifest_sha256'])
SMOKE_STATE = RUN_BASE / (RUN_NAME + '-smoke-passed.json')
SMOKE_DIR = Path(json.loads(SMOKE_STATE.read_text())['path']) if SMOKE_STATE.is_file() else None
def cli(*arguments):
    run_command([sys.executable, '-u', '-m', 'uznorm', *map(str, arguments)],
                RUN_BASE / 'logs' / (RUN_NAME + '.log'))
cli('validate-data', '--config', CONFIG_PATH)
cli('preflight', '--config', CONFIG_PATH)

## 4. Faqat 2 optimizer qadamlik smoke

Model yuklash, GPU forward/backward, sakkiz kategoriya generatsiyasi, W&B va checkpoint/export tekshiriladi.
Bu model sifati testi emas. Qayta ishga tushirilsa yangi alohida smoke papka yaratiladi.
Runtime uzilib, eski smoke muvaffaqiyatli bo‘lsa uning yo‘li 3-katakda Drive’dan tiklanadi.
`performance.json` peak GPU xotirasi va qadam vaqtini saqlaydi. Smoke’da accumulation=1:
bu vaqtni full treningga to‘g‘ridan-to‘g‘ri ko‘paytirib ETA/unit sarfi deb bo‘lmaydi.

In [ ]:
SMOKE_DIR = RUN_BASE / (RUN_NAME + '-smoke-' + uuid.uuid4().hex[:8])
cli('smoke', '--config', CONFIG_PATH, '--output', SMOKE_DIR)
performance = json.loads((SMOKE_DIR / 'performance.json').read_text(encoding='utf-8'))
print(json.dumps(performance, indent=2))
print('Smoke hisoblash tugadi. Endi quyidagi Drive saqlash testini bajaring.')

### Smoke checkpoint Drive’da qolishini tekshirish

**Faqat smoke tugagan va bu runtime’da boshqa trening ishlamayotgan paytda bajaring.**
Drive kutilayotgan yozuvlarni yuborib unmount bo‘ladi; keyin qayta ulanadi.
Checkpoint va export qayta ulangan Drive’dan hash orqali tekshiriladi. Bu qadam tugamaguncha
full trening ochilmaydi. Qayta ulanish Google ruxsat oynasini chiqarishi mumkin.
Xato bo‘lsa shu tekshiruvni tuzatamiz; yangi smoke yaratish uchun oldingi katakni takrorlash shart emas.
Bu bir martalik saqlash sinovi; kelajakdagi sessiya yoki bulut sync’i uzilmasligiga kafolat emas.

In [ ]:
from uznorm.checkpoints import latest_complete
from uznorm.io import verify_seal, write_json
guard_output(SMOKE_DIR)
if not (SMOKE_DIR / 'TRAINING_COMPLETE.json').is_file():
    raise RuntimeError('Avval smoke muvaffaqiyatli tugashi kerak.')
print('Drive yozuvlarini yuborib, qayta ulayapman. Bu katta fayllar uchun vaqt oladi.')
drive.flush_and_unmount(timeout_ms=180_000)
drive.mount('/content/drive')
guard_output(SMOKE_DIR)
if latest_complete(SMOKE_DIR) is None:
    raise RuntimeError('Qayta ulangan Drive’da butun smoke checkpoint topilmadi; full trening bloklandi.')
verify_seal(SMOKE_DIR / 'export', 'EXPORT_COMPLETE.json')
write_json(SMOKE_DIR / 'DRIVE_DURABILITY.json', {
    'storage_id': EXPECTED_STORAGE_ID, 'verified_after_flush_and_remount': True,
    'future_sync_guaranteed': False})
write_json(SMOKE_STATE, {'path': str(SMOKE_DIR)})
print('DRIVE SAQLASH TESTI O‘TDI. Smoke og‘irliklaridan full trening boshlanmaydi.')

## 5. Full fine-tuning

Avval smoke muvaffaqiyatli tugasin. Shundan keyin `RUN_FULL_TRAINING=True`.
Checkpoint’dan davom etish uchun shu config/run bilan `RESUME=True`.
W&B’da `train/loss`, `train/grad_norm`, `eval/category_macro_cer_pct`, `eval/category_*_raw_cer_pct`,
`eval/identity_change_rate_pct` va `eval/generation_missing_eos_pct`ni kuzating.
250 qadamda 1024 monitor misoli baholanadi; bu T4’da sezilarli vaqt olishi mumkin.
Yangi himoya: 1-qadamda, keyin har 50 optimizer qadamda yoki 10 daqiqa o‘tgach
keyingi optimizer qadam chegarasida va har validationdan OLDIN recovery checkpoint.
Oxirgi 2 ta recovery, odatiy oxirgi 3 ta checkpoint va kerakli best-model nusxalari saqlanadi; yangi butuni yozilgandan keyin
eski recovery nusxalari retention bo‘yicha o‘chadi. Vaqt/checkpoint sync davomiyligi kafolatlanmaydi.
Drive’da checkpoint, smoke va export uchun yetarli kvota qoldiring: Base’da ko‘plab GiB kerak;
diskdagi Colab bo‘sh joyi Drive hisobining kvotasi emas. Eski runlarni bu notebook o‘chirmaydi.
Learning rate yoki batchni ishlayotgan run ichida almashtirmang. OOM bo‘lsa yangi config/run/smoke.

In [ ]:
RUN_FULL_TRAINING = False
RESUME = (RUN_DIR / 'run-meta.json').is_file()
CONFIRM_OLD_PROCESS_STOPPED = False  # Faqat eski runtime/jarayon tugaganini tekshirgach True.
if RUN_FULL_TRAINING:
    guard_output(RUN_DIR)
    if SMOKE_DIR is None:
        raise RuntimeError('Avval smoke katagini bajaring.')
    if RESUME and not (RUN_DIR / 'TRAINING_COMPLETE.json').is_file():
        from uznorm.checkpoints import latest_complete
        print('Checkpoint hashlarini tekshiryapman; katta fayllar sabab vaqt olishi mumkin.')
        checkpoint = latest_complete(RUN_DIR)
        if checkpoint is None:
            raise RuntimeError('Butun checkpoint topilmadi. Eski run saqlansin; noldan boshlash uchun yangi RUN_NAME ni ongli tanlang.')
        print('DAVOM ETISH:', checkpoint)
        lock = RUN_DIR / '.run.lock'
        if lock.exists():
            if not CONFIRM_OLD_PROCESS_STOPPED:
                raise RuntimeError('Lock qolgan. Eski jarayon ishlamayotganini tekshiring; keyin CONFIRM_OLD_PROCESS_STOPPED=True.')
            guard_output(RUN_DIR)
            if lock.resolve().parent != RUN_DIR.resolve() or lock.is_symlink():
                raise RuntimeError('Noto‘g‘ri lock yo‘li; o‘chirilmaydi.')
            lock.unlink()  # Only the explicitly confirmed stale lock, never a checkpoint.
            print('Faqat eski .run.lock olib tashlandi; checkpointlar saqlandi.')
    arguments = ['train', '--config', CONFIG_PATH, '--output', RUN_DIR, '--smoke-run', SMOKE_DIR]
    if RESUME:
        arguments.append('--resume')
    cli(*arguments)
else:
    print('Full trening o‘chirilgan. Tayyor bo‘lsangiz RUN_FULL_TRAINING=True qiling.')

## 6. To‘liq validation va inson bahosi

16 708 validation misoli. Raw prediction va 200 ta review namunasi faqat Drive’da saqlanadi.
`manual-review-200.jsonl`ning nusxasini alohida faylga olib tahrirlang; asl report hash bilan himoyalangan.
CER/F1 semantic meaning accuracy emas. Ma’no, grammatika va entity saqlanishini inson tekshirishi kerak.

In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    cli('evaluate', '--config', CONFIG_PATH, '--output', RUN_DIR, '--split', 'validation')

## 7. Yakuniy test — model va sozlamalar muzlatilgandan keyingina

Test natijasiga qarab shu testga moslab parametr tanlamang. Keyingi iteratsiyalar yangi holdout talab qilishi mumkin.

In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    cli('evaluate', '--config', CONFIG_PATH, '--output', RUN_DIR, '--split', 'test', '--accept-test')

## 8. Tayyor export bilan inference

Natija — model taklifi. Maxsus entity/meaning guard hali kafolatlangan emas; ism, raqam, inkor va ma’noni ko‘rib chiqing.

In [ ]:
RUN_INFERENCE = False
if RUN_INFERENCE:
    from uznorm.inference import Normalizer
    normalizer = Normalizer(RUN_DIR / 'export')
    print(normalizer.correct('men bugun maktabga dostim bln bordim'))